In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-stage-3-2026")

print("Path to dataset files:", path)

In [ ]:
# Write your code here
import os
import torch
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

# # 1. Define Transforms
train_transform = transforms.Compose([
    transforms.Resize((32,32)),
    transforms.RandomRotation(15),
    transforms.ToTensor()
])
# 1. Define Transforms this for test
test_transform = transforms.Compose([
    transforms.Resize((32,32)),
    transforms.ToTensor()
])

# path of folder i will use it bellow
data_root = os.path.join(path, "PlantVillage")

# create Datasets
train_dataset = ImageFolder(os.path.join(data_root,"train"), transform=train_transform)
test_dataset  = ImageFolder(os.path.join(data_root,"test"), transform=test_transform)

# crate dataLoaders and split to train and test
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

# here i disply some images
images, labels = next(iter(train_loader))
classes = train_dataset.classes

fig, axes = plt.subplots(1,5, figsize=(12,3))
for i in range(5):
    img = images[i].permute(1,2,0)
    axes[i].imshow(img)
    axes[i].set_title(classes[labels[i]])
    axes[i].axis("off")

plt.show()


In [ ]:
# Write your code here
import torch.nn as nn
import torch.nn.functional as F

# Define the CNN Model i change the name of cnn
class PotatoCNN(nn.Module):
    def __init__(self):
        super().__init__()

        # Convolutional Layers 5 layers and i apply BN after each lyer
        self.conv1 = nn.Conv2d(3,16,3,padding=1)
        self.bn1 = nn.BatchNorm2d(16)

        self.conv2 = nn.Conv2d(16,32,3,padding=1)
        self.bn2 = nn.BatchNorm2d(32)

        self.conv3 = nn.Conv2d(32,64,3,padding=1)
        self.bn3 = nn.BatchNorm2d(64)

        self.conv4 = nn.Conv2d(64,128,3,padding=1)
        self.bn4 = nn.BatchNorm2d(128)

        self.conv5 = nn.Conv2d(128,256,3,padding=1)
        self.bn5 = nn.BatchNorm2d(256)
        # Pooling Layer
        self.pool = nn.MaxPool2d(2,2)

        # Fully Connected Layers
        self.fc1 = nn.Linear(256*1*1, 128)
        self.fc2 = nn.Linear(128, 3)

    def forward(self,x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))         # Activation  Relu i use it in forward
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = self.pool(F.relu(self.bn4(self.conv4(x))))
        x = self.pool(F.relu(self.bn5(self.conv5(x))))

        x = torch.flatten(x,1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x



In [ ]:
# Write your code here
from tqdm import tqdm # Shows progress bar

# Training loop
def train_one_epoch(model,loader,criterion,optimizer,device):
    model.train() # Set model to training mode
    loss_total=0
    correct=0
    total=0

    for images,labels in tqdm(loader):
        images,labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs=model(images) # Forward pass
        loss=criterion(outputs,labels)  # Compute loss

        loss.backward() # Backpropagation
        optimizer.step() # Update weights

        loss_total+=loss.item()
        preds=outputs.argmax(1)# Get class with probability
        correct+=(preds==labels).sum().item()
        total+=labels.size(0)

    return loss_total/len(loader),100*correct/total

# 🔹 Validation Loop
def validate(model,loader,criterion,device):
    model.eval() # Set model to evaluation mode
    loss_total=0
    correct=0
    total=0

    with torch.no_grad(): # Disable gradient computation
        for images,labels in loader:
            images,labels = images.to(device), labels.to(device)
            outputs=model(images) # Forward pass
            loss=criterion(outputs,labels)  # Compute loss
           # Compute accuracy
            loss_total+=loss.item()
            preds=outputs.argmax(1) # Get predicted class
            correct+=(preds==labels).sum().item()
            total+=labels.size(0)

    return loss_total/len(loader),100*correct/total



In [ ]:
# Write your code here
import torch.optim as optim

device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

model=PotatoCNN().to(device)  # define model
criterion=nn.CrossEntropyLoss() # define loss CrossEntropyLoss
optimizer=optim.Adam(model.parameters(),lr=0.001)  # and optimizer Adam

num_epochs=10
# Lists to store metrics
train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

# Training process
for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate(model, test_loader, criterion, device)

    # Store metrics
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)

    print(f"Epoch [{epoch+1}/{num_epochs}] | "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}% | "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")

# plots
plt.subplot(1,2,1)
plt.plot(train_losses,label="train")
plt.plot(val_losses,label="val")
plt.legend()
plt.title("Loss")

plt.subplot(1,2,2)
plt.plot(train_accuracies,label="train")
plt.plot(val_accuracies,label="val")
plt.legend()
plt.title("Accuracy")

plt.show()



In [ ]:
# Write your code here
class ResidualPotatoCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # same lyers before 5
        self.conv1=nn.Conv2d(3,16,3,padding=1)
        self.conv2=nn.Conv2d(16,32,3,padding=1)
        self.conv3=nn.Conv2d(32,64,3,padding=1)
        self.conv4=nn.Conv2d(64,64,3,padding=1)
        self.conv5=nn.Conv2d(64,128,3,padding=1)

        self.pool=nn.MaxPool2d(2,2)

        # i try use summation but i know concat is better i know just summation
        self.res_conv = nn.Conv2d(32,64,1)

        self.fc=nn.Linear(128,3)

    def forward(self,x):
        x=self.pool(F.relu(self.conv1(x)))

        res=self.pool(F.relu(self.conv2(x)))

        x=self.pool(F.relu(self.conv3(res)))
        x=self.pool(F.relu(self.conv4(x)))

        # here i match residual to x for summation
        res=self.pool(self.pool(res))
        res=self.res_conv(res)

        x=x+res  #  summation

        x=self.pool(F.relu(self.conv5(x)))

        x=torch.flatten(x,1)
        return self.fc(x)


# Training process like before
model=ResidualPotatoCNN().to(device)
optimizer=optim.Adam(model.parameters(),lr=0.001)

# Lists to store metrics
train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

# Training process
for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate(model, test_loader, criterion, device)

    # Store metrics
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)

    print(f"Epoch [{epoch+1}/{num_epochs}] | "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}% | "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")

plt.subplot(1,2,1)
plt.plot(train_losses); plt.plot(val_losses); plt.title(" Loss")
plt.subplot(1,2,2)
plt.plot(train_accuracies); plt.plot(val_accuracies); plt.title(" Acc")
plt.show()
